In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import mpmath as mp
from tqdm.auto import tqdm
%matplotlib widget

In [ ]:
nb = 1e-3
ns = 1e-6
N = 200
etamax = 1e3
etavals = np.linspace(0,etamax,N)
eta1,eta2 = np.meshgrid(etavals,etavals)

In [ ]:
def nu0(eta,nb,ns):
    to_sq = (1-eta)*ns +1+nb
    to_sqrt = to_sq**2 + 4*eta*nb*ns
    return mp.sqrt(to_sqrt)
def nu1(eta,nb,ns):
    diff = nb - (1-eta) *ns
    return diff + nu0(eta,nb,ns)
def nu2(eta,nb,ns):
    diff = (1-eta) *ns - nb
    return diff + nu0(eta,nb,ns)
def lamsq(eta,nb,ns):
    to_sqrt = eta*ns*(1+ns)
    num = 1+nb+ns+eta*ns + 2*mp.sqrt(to_sqrt)
    return num/nu0(eta,nb,ns)
def arccoth(x):
    return mp.acoth(x)

In [ ]:
@np.vectorize
def Qrel(eta1,eta2,nb,ns):
# This freaks out with very small values...
# Could be a situation where I should redo this with mpmath to avoid such extreme values
# but it could also be worth reworking this to use logs instead of arctanh, which may simplify things considerably
# mpmath doesn't seem to fix the issue, which is somewhat surprising.
    eta1 = mp.mpf(eta1)
    eta2 = mp.mpf(eta2)
    nb = mp.mpf(nb)
    ns = mp.mpf(ns)

    # Define values that will be used
    nu11 = nu1(eta1,nb,ns)
    nu12 = nu1(eta2,nb,ns)
    nu21 = nu2(eta1,nb,ns)
    nu22 = nu2(eta2,nb,ns)

    
    logterm = mp.log(1+nb-eta2) - mp.log(1+nb-eta1)
    constterm = 2*(nu11 *arccoth(nu11) + nu21 * arccoth(nu21))
    sumterm = (nu11 + nu21) *(arccoth(nu12) + arccoth(nu22))/2
    lamterm = lamsq(eta1,nb,ns)/lamsq(eta2,nb,ns) + lamsq(eta2,nb,ns)/lamsq(eta1,nb,ns)
    diffterm = (nu11 - nu21) *(arccoth(nu12) - arccoth(nu22))
    nonlogterm = sumterm*lamterm + diffterm-constterm
    return (logterm+nonlogterm)/2

In [ ]:
qrels = Qrel(eta1,eta2,nb,ns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,qrels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Quantum Relative Entropy')
ax.set_title("TMSV Quantum Limit")
plt.show()

In [ ]:
def rel_ent_pnr(eta1,eta2,nb,ns):
    def P0I(eta,nb,ns):
        return (1+ns)*(1+nb)
    def P0II(eta,nb,ns):
        return 1+ eta*ns+nb
    def P0SI(eta,nb,ns):
        return 1+ns
    def P0(eta,nb,ns):
        return 1/P0I(eta,nb,ns)
    def PI(eta,nb,ns):
        return 1/P0II(eta,nb,ns)- P0(eta,nb,ns)
    def PS(eta,nb,ns):
        return 1/P0SI(eta,nb,ns)- P0(eta,nb,ns)
    def PC(eta,nb,ns):
        return 1-PS(eta,nb,ns)-PI(eta,nb,ns)-P0(eta,nb,ns)
    # For our calculation here we can ignore the PS and P0 terms since they have no eta dependance
    PIval = PI(eta1,nb,ns) * (np.log(PI(eta1,nb,ns)) - np.log(PI(eta2,nb,ns)))
    PCval = PC(eta1,nb,ns) * (np.log(PC(eta1,nb,ns)) - np.log(PC(eta2,nb,ns)))
    return PIval +PCval

In [ ]:
pnr_rels = rel_ent_pnr(eta1,eta2,nb,ns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,pnr_rels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("Click No-Click Detection")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
pnr_surf = ax.plot_surface(eta1,eta2,pnr_rels, label="Click No-Click")
ql_surf = ax.plot_surface(eta1,eta2,qrels, label="Quantum Limit")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("Click No-Click Detection vs Quantum Limit")
ax.legend(handles = [pnr_surf,ql_surf])
plt.show()

In [ ]:
def rel_ent_su11(eta1,eta2,nb,ns):
    def P0I(eta):
        return nb*(1+2*ns) + (1+(1+np.sqrt(eta))*ns)**2
    def P0II(eta):
        return 1+ nb*(1+ns)+ ns*(1+ns)*(1+np.sqrt(eta))**2
    def P0SI(eta):
        return 1+ns*(2+nb+ns+2*(1+ns)*np.sqrt(eta)+ns*eta)
    def P0(eta):
        return 1/P0I(eta)
    def PI(eta):
        return 1/P0II(eta)- P0(eta)
    def PS(eta):
        return 1/P0SI(eta)- P0(eta)
    def PC(eta):
        return 1-PS(eta)-PI(eta)-P0(eta)
    
    P0val = P0(eta1) * (np.log(P0(eta1)) - np.log(P0(eta2)))
    PSval = PS(eta1) * (np.log(PS(eta1)) - np.log(PS(eta2)))
    PIval = PI(eta1) * (np.log(PI(eta1)) - np.log(PI(eta2)))
    PCval = PC(eta1) * (np.log(PC(eta1)) - np.log(PC(eta2)))
    return PIval +PCval+PSval+P0val

In [ ]:
su11_rels = rel_ent_su11(eta1,eta2,nb,ns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,su11_rels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_title("SU(1,1) Detection")
ax.set_zlabel('Relative Entropy')
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
su11_surf = ax.plot_surface(eta1,eta2,su11_rels, label = "SU(1,1)")
pnr_surf = ax.plot_surface(eta1,eta2,pnr_rels, label = "Click No-Click")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("SU(1,1) vs Click No-Click")
ax.legend(handles = [su11_surf,pnr_surf])
plt.show()

In [ ]:

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
pnr_surf = ax.plot_surface(eta1,eta2,pnr_rels, label="SU(1,1)")
ql_surf = ax.plot_surface(eta1,eta2,qrels, label="Quantum Limit")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("SU(1,1) vs Quantum Limit")
ax.legend(handles = [pnr_surf,ql_surf])
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,(su11_rels - pnr_rels)/su11_rels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Fractional relative entropy improvement')
plt.show()

In [ ]:
def rel_ent_coh_mp_single(eta1,eta2,nb,ns):
    A = 1/(1+nb)
    B = A/2
    C = 1/ns
    D = C/2
    def P0(eta,r):
        #return mp.exp(-r**2*eta/(2+2*nb))/(1+nb)
        return mp.exp(-r**2*eta*B)*A
    def rel_ent(r):
        P01 = P0(eta1,r)
        P02 = P0(eta2,r)
        PC1 = 1-P01
        PC2 = 1-P02
        # PCval is a bit of a lost cause, but P0val can certainly be calculated more efficiently
        # P0val = -r**2*P01 *(eta1-eta2)*B
        # Could potentially work, but it doesn't buy us much computationally
        # The real work seems to be the numeric integration which this doesn't let us eliminate
        # In order to do that we would need to calculate the second term explicitly as well
        # as the first (the first has a form that looks very analytically calculatable)
        P0val = P01*(mp.log(P01) - mp.log(P02))
        PCval = PC1*(mp.log(PC1)-mp.log(PC2))
        res = P0val +PCval
        #res[np.isnan(res)] = 0
        return res
    def to_int(r):
        #return mp.exp(-r**2/(2*ns)) *r/ns *rel_ent(r)
        return mp.exp(-r**2*D) *r*C *rel_ent(r)
    res = mp.quad(to_int,[0,mp.inf])
    return res
def rel_ent_coh_mp(eta1,eta2,nb,ns):
    shape = np.shape(eta1)
    to_ret = np.zeros(shape)
    for index in tqdm(np.ndindex(shape),total=np.prod(shape)):
        eta1v = eta1[index]
        eta2v = eta2[index]
        to_ret[index] = rel_ent_coh_mp_single(eta1v,eta2v,nb,ns)
    return to_ret

In [ ]:
# Use the first option to generate the data, and the second to load it from a file

#coh_rels = rel_ent_coh_mp(eta1,eta2,nb,ns)

#data = np.load("../_data/coh_ent.npz")
#coh_rels = data['rel_ent']

In [ ]:
rel_ent_coh_mp_single(.1,.2,nb,ns)

In [ ]:
#fig = plt.figure()
#ax = fig.add_subplot(projection='3d')
#ax.plot_surface(eta1,eta2,coh_rels)
#ax.set_xlabel('Transmisivity 1')
#ax.set_ylabel('Transmisivity 2')
#ax.set_zlabel('Relative Entropy')
#ax.set_title("Coherent Light")
#plt.show()

In [ ]:
# For saving data

#np.savez_compressed("../_data/coh_ent",eta1=eta1,eta2=eta2,nb=nb,ns=ns,rel_ent=coh_rels)

In [ ]:
#fig = plt.figure()
#ax = fig.add_subplot(projection='3d')
#su11_surf = ax.plot_surface(eta1,eta2,su11_rels, label = "SU(1,1)")
#coh_surf = ax.plot_surface(eta1,eta2,coh_rels, label = "Coherent")
#ax.set_xlabel('Transmisivity 1')
#ax.set_ylabel('Transmisivity 2')
#ax.set_zlabel('Relative Entropy')
#ax.set_title("SU(1,1) vs Coherent")
#ax.legend(handles = [su11_surf,coh_surf])
#plt.show()

In [ ]:
def Qrel_coh(eta1,eta2,nb,ns):
    return ns*(eta1-eta2)**2 *(np.log(1+nb) - np.log(nb))

In [ ]:
coh_qrels = Qrel_coh(eta1,eta2,nb,ns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,coh_qrels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Quantum Relative Entropy')
ax.set_title("Coherent Quantum Limit")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
tmsv_surf = ax.plot_surface(eta1,eta2,qrels, label = "TMSV")
coh_surf = ax.plot_surface(eta1,eta2,coh_qrels, label = "Coherent")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Quantum Relative Entropy')
ax.set_title("TMSV vs Coherent, Quantum Limit")
ax.legend(handles = [tmsv_surf,coh_surf])
plt.show()

In [ ]:
# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')
# q_surf = ax.plot_surface(eta1,eta2,coh_qrels, label = "Quantum Limit")
# coh_surf = ax.plot_surface(eta1,eta2,coh_rels, label = "Click No-Click")
# ax.set_xlabel('Transmisivity 1')
# ax.set_ylabel('Transmisivity 2')
# ax.set_zlabel('Relative Entropy')
# ax.set_title("Coherent Light Click No-Click vs Quantum Limit")
# ax.legend(handles = [q_surf, coh_surf])
# plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
su11_surf = ax.plot_surface(eta1,eta2,su11_rels, label = "SU(1,1)")
coh_surf = ax.plot_surface(eta1,eta2,coh_qrels, label = "Coherent")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("SU(1,1) vs Coherent Quantum Limit")
ax.legend(handles = [su11_surf,coh_surf])
plt.show()

In [ ]:
def rel_ent_homo(eta1, eta2, nb, ns):
    return 2*(eta1 - eta2)**2*ns *np.sqrt(np.pi/(1+nb))

In [ ]:
homo_rels = rel_ent_homo(eta1,eta2,nb,ns)

In [ ]:

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,homo_rels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("Coherent Light Homodyne")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
homo_surf = ax.plot_surface(eta1,eta2,homo_rels, label = "Homodyne")
coh_surf = ax.plot_surface(eta1,eta2,coh_qrels, label = "Coherent")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("Coherent, Homodyne vs Quantum Limit")
ax.legend(handles = [homo_surf,coh_surf])
plt.show()

In [ ]:

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
su11_surf = ax.plot_surface(eta1,eta2,su11_rels, label = "SU(1,1)")
coh_surf = ax.plot_surface(eta1,eta2,homo_rels, label = "Coherent")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("SU(1,1) vs Coherent Homodyne")
ax.legend(handles = [su11_surf,coh_surf])
plt.show()